In [ ]:
import os

# change working directory so that it picks up the grapehelper library
os.chdir("/home/ftorgano/rna-kg-analysis")
print(os.getcwd())

In [ ]:
import importlib
import logging

import helper_lib.graph
import helper_lib.cache
import helper_lib.predict

importlib.reload(helper_lib)
importlib.reload(helper_lib.graph)
importlib.reload(helper_lib.cache)
importlib.reload(helper_lib.predict)
helper_lib.cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")
logging.basicConfig(level=logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
import pandas as pd

In [ ]:
view_number = 9

In [ ]:
view_undirected_rnakg = helper_lib.graph.load_view_rnakg(view_number,directed=False)

In [ ]:
view_undirected_rnakg

In [ ]:
df_view = helper_lib.graph.build_triples_df(view_undirected_rnakg)

# Running predictions

## LINE

In [ ]:
from grape.embedders import FirstOrderLINEEnsmallen
from grape.edge_prediction import DecisionTreeEdgePrediction, RandomForestEdgePrediction

seed = 42

model_tree = DecisionTreeEdgePrediction(
    edge_embedding_methods = 'Concatenate',
    use_scale_free_distribution = True,
    training_unbalance_rate = 1,
    max_depth=100,
    random_state=seed
)
model_forest = RandomForestEdgePrediction (
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    n_estimators=100
)
embedder_line = FirstOrderLINEEnsmallen(random_state=seed, enable_cache=False, embedding_size=10, verbose=False)

In [ ]:
# miRNA-Phenotype fails to generate negative train/test set
pairs_to_predict = [('miRNA','Phenotype'),('lncRNA','Phenotype'),('miRNA','Gene'),('miRNA','GO'),('lncRNA','GO'),('lncRNA','Disease'),('Protein','GO')]

results_fun_line_tree = helper_lib.predict.edge_pred_pairs(view_undirected_rnakg, embedder_line, model_tree, pairs_to_predict, seed=seed, clear_output=True,
                                         use_scale_free_distribution = True)
results_fun_line_tree.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_tree.csv')
results_fun_line_forest = helper_lib.predict.edge_pred_pairs(view_undirected_rnakg, embedder_line, model_forest, pairs_to_predict, seed=seed, clear_output=True,
                                          use_scale_free_distribution = True)
results_fun_line_forest.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_forest.csv')

full_results = pd.concat([results_fun_line_tree,results_fun_line_forest])
full_results.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final.csv')

In [ ]:
for elem in full_results['name'].unique():
    print(elem)
    for pair in pairs_to_predict:
        print(pair)
        df_results = full_results[full_results['name']==elem]
        results_custom_model = df_results[(df_results['Source Type']==pair[0]) & (df_results['Destination Type']==pair[1])]
        positive = results_custom_model['Positive balanced accuracy']
        print(f"positive: {positive.mean():.2%} ± {positive.std():.2%}".replace("%","\\%"))
        negative = results_custom_model['Negative balanced accuracy']
        print(f"negative: {negative.mean():.2%} ± {negative.std():.2%}".replace("%","\\%"))
        mean = results_custom_model['Mean balanced accuracy']
        print(f"mean:     {mean.mean():.2%} ± {mean.std():.2%}".replace("%","\\%"))
        print('\n')

In [ ]:
for elem in full_results['name'].unique():
    for pair in pairs_to_predict:
        df_results = full_results[full_results['name']==elem]
        model = ''.join(df_results['Model'].iloc[0].split(' ')[:2])
        results_custom_model = df_results[(df_results['Source Type']==pair[0]) & (df_results['Destination Type']==pair[1])]
        positive = results_custom_model['Positive balanced accuracy']
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%","\\%")
        negative = results_custom_model['Negative balanced accuracy']
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%","\\%")
        mean = results_custom_model['Mean balanced accuracy']
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%","\\%")
        print(f'{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} \\\\')

## Node2Vec SkipGram BFS

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
from grape.edge_prediction import DecisionTreeEdgePrediction, RandomForestEdgePrediction

seed = 42

model_tree = DecisionTreeEdgePrediction(
    edge_embedding_methods = 'Concatenate',
    use_scale_free_distribution = True,
    training_unbalance_rate = 1,
    max_depth=100,
    random_state=seed
)
model_forest = RandomForestEdgePrediction (
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    n_estimators=100
)
embedder_node2vec_bfs = Node2VecSkipGramEnsmallen(random_state=seed, return_weight=5, explore_weight=0.2, embedding_size=10, 
                          verbose=False,enable_cache=True)

In [ ]:
pairs_to_predict = [('miRNA','Phenotype'),('lncRNA','Phenotype'),('miRNA','Gene'),('miRNA','GO'),('lncRNA','GO'),('lncRNA','Disease'),('Protein','GO')]

results_fun_node2vecBFS_tree = helper_lib.predict.edge_pred_pairs(view_undirected_rnakg, embedder_node2vec_bfs, model_tree, pairs_to_predict, seed=seed, clear_output=True,
                                        use_scale_free_distribution = True)
results_fun_node2vecBFS_tree.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_tree.csv')
results_fun_node2vecBFS_forest = helper_lib.predict.edge_pred_pairs(view_undirected_rnakg, embedder_node2vec_bfs, model_forest, pairs_to_predict, seed=seed, clear_output=True,
                                          use_scale_free_distribution = True)
results_fun_node2vecBFS_forest.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_forest.csv')

full_results = pd.concat([results_fun_node2vecBFS_tree,results_fun_node2vecBFS_forest])
full_results.to_csv(f'./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_node2vecBFS.csv')

In [ ]:
for elem in full_results['name'].unique():
    print(elem)
    for pair in pairs_to_predict:
        print(pair)
        df_results = full_results[full_results['name']==elem]
        results_custom_model = df_results[(df_results['Source Type']==pair[0]) & (df_results['Destination Type']==pair[1])]
        positive = results_custom_model['Positive balanced accuracy']
        print(f"positive: {positive.mean():.2%} ± {positive.std():.2%}".replace("%","\\%"))
        negative = results_custom_model['Negative balanced accuracy']
        print(f"negative: {negative.mean():.2%} ± {negative.std():.2%}".replace("%","\\%"))
        mean = results_custom_model['Mean balanced accuracy']
        print(f"mean:     {mean.mean():.2%} ± {mean.std():.2%}".replace("%","\\%"))
        print('\n')

In [ ]:
for elem in full_results['name'].unique():
    for pair in pairs_to_predict:
        df_results = full_results[full_results['name']==elem]
        model = ''.join(df_results['Model'].iloc[0].split(' ')[:2])
        results_custom_model = df_results[(df_results['Source Type']==pair[0]) & (df_results['Destination Type']==pair[1])]
        positive = results_custom_model['Positive balanced accuracy']
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%","\\%")
        negative = results_custom_model['Negative balanced accuracy']
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%","\\%")
        mean = results_custom_model['Mean balanced accuracy']
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%","\\%")
        print(f'{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} \\\\')